# TinyML Magic Wand: train and export

This notebook validates a session-grouped four-class dataset, trains the 32×32×3 classifier, converts it to full int8, and generates the exact firmware model files.

## 1. Environment and reproducibility

Use Python 3.11 and the pinned packages in `requirements.txt`. The fixed seed makes the fixture run repeatable; real learner metrics still depend on captured data.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np

TRAINING = Path.cwd() if Path('requirements.txt').exists() else Path('training')
if not (TRAINING / 'requirements.txt').exists():
    raise RuntimeError('Run this notebook from the starter root or its training directory.')
import sys
sys.path.insert(0, str(TRAINING.resolve()))
from validate_dataset import validate_dataset
from train_magic_wand import LABELS, SEED, load_dataset, seed_everything, make_model, augment, export_int8, tflite_predictions, write_c_array, write_evaluation_artifacts

DATASET = TRAINING / 'learner_dataset.json'
OUTPUT = TRAINING / 'artifacts'
OUTPUT.mkdir(exist_ok=True)
if not DATASET.exists():
    raise FileNotFoundError('Save the Lab 2 recorder export as training/learner_dataset.json before training.')
seed_everything()
print({'tensorflow_seed': SEED, 'labels': LABELS, 'dataset': str(DATASET), 'output': str(OUTPUT)})

## 2. Validate the dataset contract

Validation checks schema, unique IDs, balanced counts, point bounds, and session-grouped splits before any training begins.

In [ ]:
report = validate_dataset(DATASET)
(OUTPUT / 'dataset-validator-report.json').write_text(json.dumps(report, indent=2) + '\n')
report

## 3. Rasterize and inspect examples

The Python rasterizer preserves stroke order as red → green → blue, matching the firmware representation.

In [ ]:
images, label_ids = load_dataset(DATASET)
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for class_index, (label, axis) in enumerate(zip(LABELS, axes)):
    image = images['train'][label_ids['train'] == class_index][0].astype('uint8')
    axis.imshow(image); axis.set_title(label); axis.axis('off')
fig.suptitle('Session-grouped training examples'); fig.tight_layout()

## 4. Train and select the best checkpoint

Only training examples are augmented. Validation and test sessions remain untouched.

In [ ]:
import tensorflow as tf
train_images, train_labels = augment(images['train'], label_ids['train'])
model = make_model()
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history = model.fit(train_images, train_labels, validation_data=(images['validation'], label_ids['validation']), epochs=30, batch_size=32, callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5, mode='max', restore_best_weights=True)], verbose=2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(history.history['accuracy'], label='train'); axes[0].plot(history.history['val_accuracy'], label='validation'); axes[0].set(title='Accuracy', xlabel='Epoch', ylabel='Accuracy'); axes[0].legend(); axes[0].grid(alpha=.2)
axes[1].plot(history.history['loss'], label='train'); axes[1].plot(history.history['val_loss'], label='validation'); axes[1].set(title='Loss', xlabel='Epoch', ylabel='Cross-entropy'); axes[1].legend(); axes[1].grid(alpha=.2)
fig.tight_layout(); fig.savefig(OUTPUT / 'learning_curves.png', dpi=150)

## 5. Convert and audit the full-int8 model

The exported model must have `int8[1,32,32,3]` input, four int8 outputs, no unsupported operators, and a size no greater than 64 KB.

In [ ]:
model_bytes = export_int8(model, images['train'], OUTPUT / 'magic_wand_int8.tflite', OUTPUT / 'saved_model')
scores, metadata, operators = tflite_predictions(model_bytes, images['test'])
metadata, sorted(operators), len(model_bytes)

## 6. Evaluate the untouched test session

Report overall accuracy, per-class recall, and the complete confusion matrix. Do not tune against this partition.

In [ ]:
from sklearn.metrics import classification_report
predicted = np.argmax(scores, axis=1)
print(classification_report(label_ids['test'], predicted, target_names=LABELS, digits=3))
host_predicted = np.argmax(model.predict(images['test'], verbose=0), axis=1)
metrics = write_evaluation_artifacts(OUTPUT, model_bytes, metadata, operators, scores, label_ids['test'], host_predicted)
metrics

## 7. Generate firmware artifacts

The C array and labels are generated together so the output order cannot drift.

In [ ]:
write_c_array(model_bytes, OUTPUT)
summary = {'modelBytes': len(model_bytes), 'sha256': metrics['modelSha256'], 'operators': sorted(operators), 'labels': list(LABELS), 'generated': ['learning_curves.png', 'confusion_matrix.png', 'metrics.json', 'parity-report.json', 'model_card.md', 'magic_wand_int8.tflite', 'magic_wand_model_data.cpp', 'magic_wand_model_data.h', 'model_labels.h']}
(OUTPUT / 'notebook_summary.json').write_text(json.dumps(summary, indent=2) + '\n')
summary

## 8. Deployment handoff

The notebook has written the complete Lab 3 evidence set to `training/artifacts/`. Copy the generated C files and labels into the firmware only after checking the tensor, operator, size, accuracy, recall, and parity gates. Then load the same `.tflite` and `metrics.json` in the Virtual Wand, run the untouched test split, and save the exported report as `training/artifacts/virtual-wand-report.json`.